In [ ]:
import os
import seaborn as sns
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
# import contextily as ctx
import warnings
warnings.filterwarnings('ignore')
import sys
# sys.path.append("/path/to/sinkhole-risk-china/code")# project code path
sys.path.append("/path/to/sinkhole-risk-china/code")# project code path
from mgtwr.sel_bw import Sel_BW
from mgtwr.model import Model
import multiprocessing as mp
import psutil
from sklearn.metrics import r2_score
from shapely.geometry import Point
from pathlib import Path


In [ ]:
ssp = "hist"  # TODO: 'hist' / 'ssp1' / 'ssp2' / 'ssp3' / 'ssp4' / 'ssp5'
ssp_time = "2020"  # TODO: '2020' / '2040' / '2060' / '2080' / '2100'

In [ ]:
resolution = "10km"  # TODO: Manually change the resolution to '0.1km' / '1km' / '3km' / '5km' / '10km' / '20km' as needed
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_base_path = os.path.join(base_path, "data")

path_name = f"Points_China_all_{resolution}"
print(path_name)
# ✅ Output directory: base_path/output/path_name
out_dir = os.path.join(base_path, "outputs", path_name, ssp, ssp_time)
print(f"out_dir:{out_dir}")
os.makedirs(out_dir, exist_ok=True)


Implementation note.

In [ ]:


# pre_data_path = os.path.join(data_base_path, "Extracted_HAVE_future", "Positive_Negative_balanced", "AllFeatures_Positive_Negative_balanced_25366_ssp5_cleaned.csv")
pre_data_path = os.path.join(data_base_path, "Extracted_HAVE_future", f"Points_China_all_{resolution}", f"AllFeatures_Points_China_all_{resolution}_{ssp}_cleaned.csv")
pre_df = pd.read_csv(pre_data_path)
print("Columns (list):")
print(pre_df.columns.tolist())

###################################################################################
# ########################### Select the required characteristics##########################################
###################################################################################
# ---- Static features ----
static_features = [
    "Distance_to_karst",
    "Depth_to_Bedrock",
    "Distance_to_Fault_m",
]

# ---- Dynamic feature prefix ----
dynamic_prefixes = ["UrbanFrac", "PopTotal", "ImperviousIndex", "LAI", "Precip", "WTD", "HDS", "Tas", "Huss"]

def pick_dynamic_col(prefix: str, ssp: str, ssp_time: str) -> str:
    # 2020 or hist: use historical column
    if ssp == "hist" or ssp_time == "2020":
        return f"{prefix}_hist_2000_2010_2020"
    # 2040/2060/2080/2100: (t-20, t) 20
    t = int(ssp_time)
    return f"{prefix}_{t-20}_{t}"

pre_features = static_features + [pick_dynamic_col(p, ssp, ssp_time) for p in dynamic_prefixes]

# ---- Check for missing columns first to avoid direct KeyError ----
missing = [c for c in pre_features if c not in pre_df.columns]
if missing:
    print("❌ Missing columns:", missing)
    print("\n✅ Available columns (preview):", pre_df.columns.tolist()[:20], "...")
    raise KeyError("Some required features are not in the CSV columns.")

pre_X = pre_df[pre_features].values
print("✅ pre_features =", pre_features)
print("✅ pre_X shape =", pre_X.shape)
###################################################################################
###################################################################################
###################################################################################

# If pre_df does not have a Disaster column, create it and fill it with 3
if 'Disaster' not in pre_df.columns:
    pre_df['Disaster'] = 3

pre_y = pre_df['Disaster'].values.reshape(-1, 1)  # (n_samples, 1)

print("The shape of the argument pre_X:", pre_X.shape)
print("Shape of dependent variable pre_y:", pre_y.shape)
# Create geographic coordinate system
geometry = [Point(lon, lat) for lon, lat in zip(pre_df['Longitude'], pre_df['Latitude'])]
pre_gdf = gpd.GeoDataFrame(pre_df, geometry=geometry, crs="EPSG:4326")

# Convert to projected coordinate system (using isometric projection)
pre_gdf = pre_gdf.to_crs("EPSG:3857")  # Web Mercator projection
pre_coords1 = np.array([(pt.x, pt.y) for pt in pre_gdf.geometry])
pre_coords = np.hstack((pre_coords1, np.zeros((pre_coords1.shape[0], 1))))
print("Shape of coordinates pre_coords:", pre_coords.shape)

print("\\n✅ pre_features preview of the first 5 lines of corresponding data:")
display(pre_df[pre_features].head())

GWR

In [ ]:
import pickle
# Processing step.
with open(os.path.join(base_path, "code", "3_gwr_model_train", "national", "GWR", "gwr_model_national_GWR.pkl"), "rb") as f:
    saved = pickle.load(f)

gwr = saved["gwr"]
gwr_results = saved.get("gwr_results", None)
bw = saved.get("bw", None)
######

In [ ]:
y_pred_gwr, params = gwr.predict(pre_coords, pre_X)
print(y_pred_gwr.shape)
print(params.shape)
print('R2 = ', r2_score(pre_y, y_pred_gwr))

In [ ]:
# #######################Truncate prediction set output (outlier handling) ####################################
y_pred_gwr_unclipped = y_pred_gwr.copy()

CLIP_MIN = -0.539341
CLIP_MAX = 1.2834

n_clipped_low = int(np.sum(y_pred_gwr_unclipped < CLIP_MIN))
n_clipped_high = int(np.sum(y_pred_gwr_unclipped > CLIP_MAX))
y_pred_gwr = np.clip(y_pred_gwr_unclipped, CLIP_MIN, CLIP_MAX)

print(y_pred_gwr.shape)
print(params.shape)
print('Manual clipping range = ', CLIP_MIN, CLIP_MAX)
print('Prediction raw-score range before clipping = ', float(np.nanmin(y_pred_gwr_unclipped)), float(np.nanmax(y_pred_gwr_unclipped)))
print('Prediction raw-score range after clipping = ', float(np.nanmin(y_pred_gwr)), float(np.nanmax(y_pred_gwr)))
print('Clipped below min count = ', n_clipped_low)
print('Clipped above max count = ', n_clipped_high)

if globals().get('pre_has_observed_labels', False):
    print('R2 = ', float(r2_score(pre_y, y_pred_gwr)))

In [ ]:
# #####################drawing########################
import numpy as np
import matplotlib.pyplot as plt

if 'plot_gwr_output_preview' not in globals():
    def plot_gwr_output_preview(values, title_prefix="GWR Raw Output", max_unique_bar=20, hist_bins=40, focus_pct=99.0):
        values = np.asarray(values, dtype=float).reshape(-1)
        valid_mask = np.isfinite(values)
        valid_values = values[valid_mask]
        if valid_values.size == 0:
            raise ValueError("There are no limited output values available for preview.")

        lower_pct = max(0.0, (100.0 - float(focus_pct)) / 2.0)
        upper_pct = min(100.0, 100.0 - lower_pct)
        score_min, score_max = np.nanpercentile(valid_values, [lower_pct, upper_pct])
        if (not np.isfinite(score_min)) or (not np.isfinite(score_max)):
            score_min = float(np.min(valid_values))
            score_max = float(np.max(valid_values))
        if score_min == score_max:
            pad = max(abs(float(score_min)) * 0.05, 1.0e-6)
            score_min -= pad
            score_max += pad

        unique_values, counts = np.unique(valid_values, return_counts=True)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

        if unique_values.size <= max_unique_bar:
            x = np.arange(unique_values.size)
            axes[0].bar(x, counts, color="#8e7376", edgecolor="white")
            axes[0].set_xticks(x)
            axes[0].set_xticklabels([f"{v:.6g}" for v in unique_values], rotation=45, ha="right")
            axes[0].set_xlabel("Raw score")
            axes[0].set_ylabel("Count")
            axes[0].set_title(f"{title_prefix}: unique-value counts")
        else:
            display_values = valid_values[(valid_values >= score_min) & (valid_values <= score_max)]
            if display_values.size == 0:
                display_values = valid_values
            axes[0].hist(display_values, bins=np.linspace(score_min, score_max, hist_bins + 1), color="#8e7376", edgecolor="white")
            axes[0].set_xlim(score_min, score_max)
            axes[0].set_xlabel("Raw score")
            axes[0].set_ylabel("Count")
            axes[0].set_title(f"{title_prefix}: histogram (99% range)")

        sorted_values = np.sort(valid_values)
        quantiles = np.linspace(0.0, 1.0, sorted_values.size)
        axes[1].plot(quantiles, sorted_values, color="#724e52", linewidth=1.5)
        axes[1].set_xlabel("Quantile")
        axes[1].set_ylabel("Raw score")
        axes[1].set_title(f"{title_prefix}: sorted curve")
        axes[1].set_ylim(score_min, score_max)
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

        print(f"{title_prefix} finite count: {valid_values.size}")
        print(f"{title_prefix} raw-score range: [{valid_values.min():.6g}, {valid_values.max():.6g}]")
        print(f"{title_prefix} 99% display range: [{score_min:.6g}, {score_max:.6g}]")
        print(f"{title_prefix} raw-score mean/std: {valid_values.mean():.6g} / {valid_values.std(ddof=0):.6g}")
        print(f"{title_prefix} unique value count: {unique_values.size}")
        if unique_values.size <= max_unique_bar:
            print("Unique raw scores:")
            for value, count in zip(unique_values, counts):
                print(f"  {value:.10g}: {count}")

plot_gwr_output_preview(y_pred_gwr, title_prefix="Predicted GWR raw outputs")


/

In [ ]:
import pickle
from pathlib import Path
from sklearn.metrics import r2_score

# ()
pkl_path = Path(f"gwr_pre_{path_name}_{ssp}_{ssp_time}_results.pkl")
r2 = r2_score(pre_y, y_pred_gwr)

saved = {
    "y_pred_gwr": y_pred_gwr,
    "params": params,
    "r2": r2,}
with pkl_path.open("wb") as f:
    pickle.dump(saved, f, protocol=pickle.HIGHEST_PROTOCOL)
